# Reactome — Curated Pathway Database

**Reactome** is a free, open-source, curated and peer-reviewed knowledgebase of biological pathways. It provides a manually curated, peer-reviewed set of biological pathways for human and 22 other species via electronic inference.

| Property | Value |
|---|---|
| URL | https://reactome.org |
| Human reactions | ~14,000 |
| Human pathways | ~2,600 |
| Current version | v92 (2024) |
| Coverage | Signaling, metabolism, gene expression, transport, DNA repair, immune system |

In [ ]:
import requests
import time
from pathlib import Path

import polars as pl

# TODO

* [x] **Ingest data**
    * [x] Connect to Reactome ContentService REST API and confirm access
    * [x] Download human pathway list (`ReactomePathways.txt`) with caching
    * [x] Download pathway hierarchy (`ReactomePathwaysRelation.txt`) with caching
    * [x] Download UniProt-to-Reactome mappings (`UniProt2Reactome.txt`) with caching
    * [x] Parse all three files into Polars DataFrames with correct dtypes
    * [x] Save to `data/` with caching
* [ ] **Explore and clean**
    * [ ] Summarise pathway counts per top-level category
    * [ ] Check hierarchy depth distribution
    * [ ] Identify pathways with most/fewest member proteins
    * [ ] Filter to reviewed (R-HSA) human pathways only
* [ ] **Analysis**
    * [ ] Build pathway hierarchy graph using NetworkX
    * [ ] Compute proteins-per-pathway distribution
    * [ ] Identify most-connected (hub) proteins across pathways
* [ ] **Visualization**
    * [ ] Tree map of top-level pathway categories by reaction count
    * [ ] Histogram of pathway sizes
* [ ] **Statistical analysis**
    * [ ] Test for pathway size distribution fit
    * [ ] Compare human vs. inferred-species pathway counts

## 1. Ingest Data

### 1.1 Connect to Reactome ContentService API

In [ ]:
REACTOME_BASE = "https://reactome.org/ContentService"
SPECIES_HUMAN = 9606  # NCBI taxon ID for Homo sapiens

def reactome_get(endpoint: str, params: dict | None = None) -> requests.Response:
    """
    Send a GET request to the Reactome ContentService API.

    Parameters
    ----------
    endpoint : str
        API path relative to the base URL (e.g. "data/pathways/top/9606").
    params : dict, optional
        Query parameters to include in the request.

    Returns
    -------
    requests.Response
        Raw response object; caller is responsible for parsing.
    """
    url = f"{REACTOME_BASE}/{endpoint}"
    resp = requests.get(url, params=params or {}, timeout=60)
    resp.raise_for_status()
    return resp

# Fetch top-level human pathways as a connectivity / sanity check
resp = reactome_get(f"data/pathways/top/{SPECIES_HUMAN}")
top_pathways = resp.json()

print(f"Top-level human pathways: {len(top_pathways)}")
print("\nSample (first 5):")
for p in top_pathways[:5]:
    print(f"  {p['stId']:>16}  {p['displayName']}")

### 1.2 Download Human Pathway List

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

PATHWAYS_URL = "https://reactome.org/download/current/ReactomePathways.txt"
PATHWAYS_PATH = DATA_DIR / "ReactomePathways.txt"

if not PATHWAYS_PATH.exists():
    print(f"Downloading {PATHWAYS_PATH.name} ...")
    with requests.get(PATHWAYS_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(PATHWAYS_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    print(f"Saved to {PATHWAYS_PATH}")
else:
    print(f"Already downloaded: {PATHWAYS_PATH}")

print(f"File size: {PATHWAYS_PATH.stat().st_size / 1_000:.1f} KB")

### 1.3 Download Pathway Hierarchy

In [ ]:
HIERARCHY_URL = "https://reactome.org/download/current/ReactomePathwaysRelation.txt"
HIERARCHY_PATH = DATA_DIR / "ReactomePathwaysRelation.txt"

if not HIERARCHY_PATH.exists():
    print(f"Downloading {HIERARCHY_PATH.name} ...")
    with requests.get(HIERARCHY_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(HIERARCHY_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    print(f"Saved to {HIERARCHY_PATH}")
else:
    print(f"Already downloaded: {HIERARCHY_PATH}")

print(f"File size: {HIERARCHY_PATH.stat().st_size / 1_000:.1f} KB")

### 1.4 Download UniProt-to-Reactome Mappings

In [ ]:
UNIPROT_URL = "https://reactome.org/download/current/UniProt2Reactome.txt"
UNIPROT_PATH = DATA_DIR / "UniProt2Reactome.txt"

if not UNIPROT_PATH.exists():
    print(f"Downloading {UNIPROT_PATH.name} ...")
    with requests.get(UNIPROT_URL, stream=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(UNIPROT_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB", end="\r")
    print(f"\nSaved to {UNIPROT_PATH}")
else:
    print(f"Already downloaded: {UNIPROT_PATH}")

print(f"File size: {UNIPROT_PATH.stat().st_size / 1_000_000:.1f} MB")

### 1.5 Parse into Polars DataFrames

In [ ]:
# ── ReactomePathways.txt ─────────────────────────────────────────────────────
# Tab-separated, no header. Columns: pathway_id, pathway_name, species
pathways = pl.read_csv(
    PATHWAYS_PATH,
    separator="\t",
    has_header=False,
    new_columns=["pathway_id", "pathway_name", "species"],
)
print("=== pathways ===")
print(f"Shape: {pathways.shape}")
print(pathways.head())

# ── ReactomePathwaysRelation.txt ─────────────────────────────────────────────
# Tab-separated, no header. Columns: parent_id, child_id
hierarchy = pl.read_csv(
    HIERARCHY_PATH,
    separator="\t",
    has_header=False,
    new_columns=["parent_id", "child_id"],
)
print("\n=== hierarchy ===")
print(f"Shape: {hierarchy.shape}")
print(hierarchy.head())

# ── UniProt2Reactome.txt ──────────────────────────────────────────────────────
# Tab-separated, no header.
# Columns: uniprot_id, reactome_id, url, pathway_name, evidence_code, species
uniprot = pl.read_csv(
    UNIPROT_PATH,
    separator="\t",
    has_header=False,
    new_columns=["uniprot_id", "reactome_id", "url", "pathway_name", "evidence_code", "species"],
    infer_schema_length=10_000,
)
print("\n=== uniprot ===")
print(f"Shape: {uniprot.shape}")
print(uniprot.head())